In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using {device} Device...')

Using cpu Device...


In [12]:
from pathlib import Path
text = Path('tiny-shakespeare.txt').read_text()
#print(text[0:])

In [51]:
class CharactTokenizer:
    def __init__(self, vocabulary):
        self.token_id_for_char = {char: token_id for token_id, char in enumerate(vocabulary)} 
        self.char_for_token_id = {token_id: char for token_id, char in enumerate(vocabulary)}
        
    @staticmethod
    def train_from_text(text):
        vocabulary = set(text)
        return CharactTokenizer(sorted(list(vocabulary)))

    def encode(self, text):
        token_ids = []
        for char in text:
            token_ids.append(self.token_id_for_char[char])
        return torch.tensor(token_ids, dtype=torch.long)

    def decode(self, token_ids):
        chars = []
        for token_id in token_ids.tolist():
            chars.append(self.char_for_token_id[token_id])
        return ''.join(chars)

    def vocabulary_size(self):
        return len(self.token_id_for_char)

In [52]:
check = CharactTokenizer.train_from_text(text)
check
#check = CharactTokenizer(text)
#check.token_id_for_char 
#check.char_for_token_id

In [53]:
tokennizer = CharactTokenizer(text)
print(tokennizer.encode('eedy green'))

tensor([1115374, 1115374, 1115349, 1115332, 1115385, 1115391, 1115383, 1115374,
        1115374, 1115390])


In [54]:
print(tokennizer.decode(tokennizer.encode('eedy green')))

eedy green


In [55]:
tokennizer.vocabulary_size()

65

In [61]:
import pprint
token = CharactTokenizer(text)
pp = pprint.PrettyPrinter(depth=4)
#pp.pprint(token.token_id_for_char)
#pp.pprint(token.char_for_token_id)

In [77]:
from torch.utils.data import Dataset, DataLoader, RandomSampler

class TokenIdsDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, pos):
        assert pos < len(self.data) - self.block_size
        x = self.data[pos : pos + self.block_size]
        y = self.data[pos + 1 : pos + 1 + self.block_size]
        return x, y

In [66]:
tokennized_text = token.encode(text)
dataset = TokenIdsDataset(tokennized_text, block_size=64)

In [69]:
x, y = dataset[0]

In [74]:
x

tensor([1111924, 1115389, 1115383, 1115375, 1115384, 1115385, 1110567, 1115389,
        1115384, 1115389, 1113189, 1115374, 1115390, 1115299, 1115393, 1115066,
        1115374, 1115334, 1115379, 1115383, 1115374, 1115385, 1115386, 1115374,
        1115385, 1115346, 1115383, 1115379, 1114962, 1115374, 1115374, 1115349,
        1115385, 1115387, 1115390, 1115332, 1115385, 1115334, 1115380, 1115383,
        1115384, 1115378, 1115374, 1115383, 1115352, 1115385, 1115378, 1115374,
        1115387, 1115383, 1115385, 1115259, 1115374, 1115385, 1115375, 1115346,
        1115374, 1115387, 1115388, 1115392, 1115393, 1115393, 1115292, 1115373])

In [75]:
y

tensor([1115389, 1115383, 1115375, 1115384, 1115385, 1110567, 1115389, 1115384,
        1115389, 1113189, 1115374, 1115390, 1115299, 1115393, 1115066, 1115374,
        1115334, 1115379, 1115383, 1115374, 1115385, 1115386, 1115374, 1115385,
        1115346, 1115383, 1115379, 1114962, 1115374, 1115374, 1115349, 1115385,
        1115387, 1115390, 1115332, 1115385, 1115334, 1115380, 1115383, 1115384,
        1115378, 1115374, 1115383, 1115352, 1115385, 1115378, 1115374, 1115387,
        1115383, 1115385, 1115259, 1115374, 1115385, 1115375, 1115346, 1115374,
        1115387, 1115388, 1115392, 1115393, 1115393, 1115292, 1115373, 1115373])

In [71]:
token.decode(x)

'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAl'

In [79]:
sampler = RandomSampler(dataset, replacement=True)
dataloader = DataLoader(dataset, batch_size=2, sampler=sampler)

In [80]:
x,y = next(iter(dataloader))

In [81]:
x.shape

torch.Size([2, 64])

In [82]:
token.decode(x[0])

't is honour:\nAnd to repair my honour lost for him,\nI here renoun'

In [83]:
token.decode(y[0])

' is honour:\nAnd to repair my honour lost for him,\nI here renounc'